## HOEMEWORK 6
The goal of this homework is to create a regression model for predicting the car fuel efficiency (column 'fuel_efficiency_mpg').

In [81]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error
import xgboost as xgb

In [82]:
df=pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv')

In [83]:
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


## Preparing dataset

In [84]:
# Check if the missing values are presented in the features.
df.isna().any()

model_year             False
origin                 False
fuel_type              False
drivetrain             False
num_doors              False
engine_displacement    False
num_cylinders          False
horsepower              True
vehicle_weight         False
acceleration            True
fuel_efficiency_mpg    False
dtype: bool

In [85]:
# Fill missing values with zero 
missingV_cols = ['horsepower', 'acceleration']
df[missingV_cols ] = df[missingV_cols ].fillna(0.0)

In [86]:
#split 
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [87]:
#Extract the target values('fuel_efficiency_mpg')
y_train =df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test =df_test['fuel_efficiency_mpg'].values

#Remove the target from features 
df_train = df_train.drop(columns=['fuel_efficiency_mpg'])
df_val = df_val.drop(columns=['fuel_efficiency_mpg'])
df_test = df_test.drop(columns=['fuel_efficiency_mpg'])

In [88]:
#Applying dictVectorizer
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

In [89]:
#Question1:Which original feature is used for splitting the data? If you use DictVectorizer, map a one-hot feature such as origin=USA back to its original column name.
#Quelle variable d'origine est utilisée par l'arbre pour séparer les données ?
tree_reg = DecisionTreeRegressor(max_depth=1)
tree_reg.fit(X_train, y_train)
feature_index = tree_reg.tree_.feature[0]

feature_name = dv.get_feature_names_out()[feature_index]

print("Feature index:", feature_index)
print("Feature used for splitting:", feature_name)

Feature index: 9
Feature used for splitting: model_year


In [90]:
#Question2:What's the RMSE of this model on the validation data?
# Train the model
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)
#The rmse
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print("Root Mean Squared Error (RMSE):", rmse)

Root Mean Squared Error (RMSE): 1.8332197358745623


In [91]:
#Question3:Which value gives the lowest validation RMSE? Consider 3 decimal places for calculating the answer.
n_estimators= [10, 50, 100, 150]
scores = {}
#Now let's train a rf
for n in n_estimators:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    # Calculate the rmseon the validation dataset
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    scores[n] = round(rmse, 3)
    print(f"n_estimators = {n}, rmse= {scores[n]}")
low_N = min(scores, key=scores.get)

print("Lowest n-estimator", low_N)
print("lowest rmse:", scores[low_N])

n_estimators = 10, rmse= 1.833
n_estimators = 50, rmse= 1.774
n_estimators = 100, rmse= 1.77
n_estimators = 150, rmse= 1.769
Lowest n-estimator 150
lowest rmse: 1.769


In [92]:
#Question4:What's the best max_depth, using the mean RMSE?
max_depth = [10, 15, 20, 25]
N = [10, 50, 100, 150]
scores = {}

for m in max_depth:
    rmses = []

    for n in N:
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=m,
            random_state=1,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)

        # Calculate RMSE on validation dataset
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))

        # Store each RMSE
        rmses.append(rmse)

    # Calculate mean RMSE for each max_depth
    scores[m] = np.mean(rmses)

    print(f"max_depth = {m}, mean RMSE = {scores[m]:.3f}")

# Find the max_depth with the lowest mean RMSE
best_depth = min(scores, key=scores.get)

print("Best max_depth:", best_depth)
print(f"Lowest mean RMSE: {scores[best_depth]:.3f}")


max_depth = 10, mean RMSE = 1.754
max_depth = 15, mean RMSE = 1.784
max_depth = 20, mean RMSE = 1.787
max_depth = 25, mean RMSE = 1.786
Best max_depth: 10
Lowest mean RMSE: 1.754


In [93]:

# Question 5: What's the most important feature?

rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1,n_jobs=-1)
rf.fit(X_train, y_train)

features = ['vehicle_weight','horsepower','acceleration','engine_displacement']
importances = dict(zip(dv.get_feature_names_out(),rf.feature_importances_))
for feature in features:
    print(feature, round(importances[feature], 4))
print("Most important feature:", max(features, key=importances.get))



vehicle_weight 0.2033
horsepower 0.0783
acceleration 0.055
engine_displacement 0.0613
Most important feature: vehicle_weight


In [96]:
#Question6: Which eta leads to the best RMSE score on the validation dataset?
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

watchlist = [(dtrain, 'train'), (dval, 'val')]

In [97]:
etas = [0.3, 0.1]
scores = {}

for eta in etas:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1
    }

    history = {}

    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=100,
        evals=watchlist,
        evals_result=history,
        verbose_eval=False
    )

    # RMSE after 100 boosting rounds
    rmse = history['val']['rmse'][-1]
    scores[eta] = rmse

    print(f"eta = {eta}, RMSE = {rmse:.4f}")

# Find the best eta
best_eta = min(scores, key=scores.get)

print("Best eta:", best_eta)
print("Lowest RMSE:", scores[best_eta])


eta = 0.3, RMSE = 1.8265
eta = 0.1, RMSE = 1.7248
Best eta: 0.1
Lowest RMSE: 1.7248100768778967
